<!--nav--> [🗺 Learning path](README.md) · **20/41** · ◀ [Simple MultiGPU ImageClassification](./Simple_MultiGPU_ImageClassification.ipynb) · [From Fine-Tune to Production](./From_FineTune_To_Production.ipynb) ▶

# Simple Multi-GPU Audio Training

Fine-tune Whisper for speech recognition on multiple GPUs.

- **Model:** Whisper-small (244M params) — fits on free GPUs
- **Method:** LoRA + DeepSpeed ZeRO-2 + Accelerate
- **Data:** Google FLEURS (English, 500 examples) — freely available
- **Task:** Speech-to-text transcription
- **Platform:** Kaggle 2x T4 (free) or Colab T4 (free)

In [ ]:
!pip install -q transformers datasets peft accelerate deepspeed jiwer

In [ ]:
import torch, os, json, time

assert torch.cuda.is_available(), "GPU required!"

NUM_GPUS = torch.cuda.device_count()
for i in range(NUM_GPUS):
    name = torch.cuda.get_device_name(i)
    mem = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"  GPU {i}: {name} ({mem:.0f} GB)")
print(f"\nTotal GPUs: {NUM_GPUS}")

## Write Configs

In [ ]:
# DeepSpeed ZeRO-2 config
ds_config = {
    "bf16": {"enabled": True},
    "zero_optimization": {
        "stage": 2,
        "offload_optimizer": {"device": "cpu", "pin_memory": True},
        "allgather_partitions": True,
        "allgather_bucket_size": 2e8,
        "overlap_comm": True,
        "reduce_scatter": True,
        "reduce_bucket_size": 2e8,
        "contiguous_gradients": True,
    },
    "gradient_accumulation_steps": "auto",
    "gradient_clipping": "auto",
    "train_batch_size": "auto",
    "train_micro_batch_size_per_gpu": "auto",
}
with open("ds_config.json", "w") as f:
    json.dump(ds_config, f, indent=2)

# Accelerate config — accelerate-managed DeepSpeed
accel_yaml = f"""compute_environment: LOCAL_MACHINE
distributed_type: DEEPSPEED
deepspeed_config:
  gradient_accumulation_steps: auto
  gradient_clipping: auto
  offload_optimizer_device: cpu
  offload_param_device: none
  zero3_init_flag: false
  zero_stage: 2
machine_rank: 0
main_process_ip: null
main_process_port: null
main_training_function: main
mixed_precision: bf16
num_machines: 1
num_processes: {NUM_GPUS}
use_cpu: false
"""
accel_dir = os.path.expanduser("~/.cache/huggingface/accelerate")
os.makedirs(accel_dir, exist_ok=True)
with open(os.path.join(accel_dir, "default_config.yaml"), "w") as f:
    f.write(accel_yaml)

print(f"Configs written. {NUM_GPUS} GPU(s), ZeRO-2, bf16.")

## Write Training Script

Fine-tune Whisper-small on speech transcription with LoRA. Whisper is an encoder-decoder model — we apply LoRA to the decoder's attention layers.

In [ ]:
%%writefile train_audio.py
"""Distributed audio fine-tuning: Whisper-small + LoRA + DeepSpeed."""
import torch, os, json, time
os.environ["WANDB_DISABLED"] = "true"

from dataclasses import dataclass
from typing import Any, Dict, List, Union
from datasets import load_dataset, Audio
from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor,
    TrainingArguments,
    Trainer,
    TrainerCallback,
)
from peft import LoraConfig, get_peft_model

MODEL = "openai/whisper-small"

# Load model + processor
processor = WhisperProcessor.from_pretrained(MODEL)
model = WhisperForConditionalGeneration.from_pretrained(MODEL, dtype=torch.bfloat16)
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
total_params = sum(p.numel() for p in model.parameters())

# LoRA on decoder attention
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
))
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
model.print_trainable_parameters()

# Load Google FLEURS English — freely available, no auth needed
dataset = load_dataset("google/fleurs", "en_us", split="train")
# Use up to 500 examples
if len(dataset) > 500:
    dataset = dataset.select(range(500))
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))
print(f"Dataset: {len(dataset)} audio examples")


def preprocess(batch):
    audio = batch["audio"]
    inputs = processor(
        audio["array"], sampling_rate=16000,
        return_tensors="np",
    )
    batch["input_features"] = inputs.input_features[0]
    # FLEURS uses "transcription" field (not "sentence")
    batch["labels"] = processor.tokenizer(batch["transcription"]).input_ids
    return batch


dataset = dataset.map(preprocess, remove_columns=dataset.column_names)
print(f"Processed: {len(dataset)} examples")


# Data collator — pads labels, converts features to tensors
@dataclass
class WhisperDataCollator:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Pad input features
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # Pad labels
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        # Remove BOS token if present (Whisper adds it during generation)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch


# Metrics callback
class MetricsCallback(TrainerCallback):
    def __init__(self):
        self.logs = []
        self.step_times = []
        self.train_start = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.train_start = time.time()
        torch.cuda.reset_peak_memory_stats()

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            self.logs.append({
                "step": state.global_step,
                "loss": logs["loss"],
                "learning_rate": logs.get("learning_rate", 0),
                "epoch": logs.get("epoch", 0),
            })

    def on_step_end(self, args, state, control, **kwargs):
        self.step_times.append(time.time())

    def on_train_end(self, args, state, control, **kwargs):
        if int(os.environ.get("LOCAL_RANK", 0)) != 0:
            return
        train_time = time.time() - self.train_start
        gpu_mem_allocated = torch.cuda.max_memory_allocated() / 1e9
        gpu_mem_reserved = torch.cuda.max_memory_reserved() / 1e9
        num_gpus = int(os.environ.get("WORLD_SIZE", 1))
        total_steps = state.global_step
        total_samples = total_steps * args.per_device_train_batch_size * args.gradient_accumulation_steps * num_gpus

        if len(self.step_times) > 2:
            durations = [self.step_times[i+1] - self.step_times[i] for i in range(len(self.step_times)-1)]
            avg_step_time = sum(durations) / len(durations)
        else:
            avg_step_time = train_time / max(total_steps, 1)

        # ~5 sec avg audio at 16kHz = 80000 samples per example
        audio_hours = total_samples * 5 / 3600

        metrics = {
            "model": MODEL,
            "task": "speech-to-text",
            "total_params": total_params,
            "trainable_params": trainable_params,
            "trainable_pct": trainable_params / total_params * 100,
            "num_gpus": num_gpus,
            "gpu_name": torch.cuda.get_device_name(0),
            "gpu_mem_allocated_gb": round(gpu_mem_allocated, 2),
            "gpu_mem_reserved_gb": round(gpu_mem_reserved, 2),
            "gpu_mem_total_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1),
            "train_time_sec": round(train_time, 1),
            "total_steps": total_steps,
            "total_samples": total_samples,
            "audio_hours_processed": round(audio_hours, 2),
            "samples_per_sec": round(total_samples / train_time, 2),
            "avg_step_time_sec": round(avg_step_time, 3),
            "final_loss": self.logs[-1]["loss"] if self.logs else None,
            "loss_history": self.logs,
            "batch_size_per_gpu": args.per_device_train_batch_size,
            "grad_accum_steps": args.gradient_accumulation_steps,
            "effective_batch_size": args.per_device_train_batch_size * args.gradient_accumulation_steps * num_gpus,
            "learning_rate": args.learning_rate,
        }
        with open("training_metrics.json", "w") as f:
            json.dump(metrics, f, indent=2)
        print(f"\nMetrics saved to training_metrics.json")


metrics_cb = MetricsCallback()

# Train
trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="./whisper_output",
        num_train_epochs=2,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=1e-4,
        warmup_steps=20,
        logging_steps=1,
        bf16=True,
        gradient_checkpointing=True,
        report_to="none",
        save_strategy="no",
        remove_unused_columns=False,
        label_names=["labels"],
    ),
    train_dataset=dataset,
    data_collator=WhisperDataCollator(processor=processor),
    callbacks=[metrics_cb],
)

trainer.train()
trainer.save_model("./whisper_output/final")
processor.save_pretrained("./whisper_output/final")

if int(os.environ.get("LOCAL_RANK", 0)) == 0:
    print("\n" + "=" * 50)
    print("Whisper audio training complete!")
    print("=" * 50)

## Launch Training

In [ ]:
print(f"Launching audio training on {NUM_GPUS} GPU(s)...")
start = time.time()

!accelerate launch --num_processes={NUM_GPUS} train_audio.py

elapsed = time.time() - start
print(f"\nDone! {elapsed:.0f}s on {NUM_GPUS}x {torch.cuda.get_device_name(0)}")

## Performance Dashboard

In [ ]:
import json, matplotlib.pyplot as plt, matplotlib.ticker as ticker
from IPython.display import HTML, display

with open("training_metrics.json") as f:
    m = json.load(f)

# --- Loss Curve + Learning Rate ---
fig, ax1 = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor("#0d1117")
ax1.set_facecolor("#0d1117")

steps = [h["step"] for h in m["loss_history"]]
losses = [h["loss"] for h in m["loss_history"]]
lrs = [h["learning_rate"] for h in m["loss_history"]]

color_loss = "#58a6ff"
ax1.plot(steps, losses, color=color_loss, linewidth=2, label="Loss", zorder=3)
ax1.fill_between(steps, losses, alpha=0.1, color=color_loss)
ax1.set_xlabel("Step", color="#8b949e", fontsize=11)
ax1.set_ylabel("Loss", color=color_loss, fontsize=11)
ax1.tick_params(axis="y", labelcolor=color_loss)
ax1.tick_params(axis="x", colors="#8b949e")
ax1.grid(True, alpha=0.15, color="#30363d")
ax1.spines["top"].set_visible(False)
for spine in ax1.spines.values():
    spine.set_color("#30363d")

ax2 = ax1.twinx()
color_lr = "#f0883e"
ax2.plot(steps, lrs, color=color_lr, linewidth=1.5, linestyle="--", alpha=0.7, label="LR")
ax2.set_ylabel("Learning Rate", color=color_lr, fontsize=11)
ax2.tick_params(axis="y", labelcolor=color_lr)
ax2.spines["top"].set_visible(False)
for spine in ax2.spines.values():
    spine.set_color("#30363d")
ax2.yaxis.set_major_formatter(ticker.FormatStrFormatter("%.1e"))

fig.suptitle("Whisper Training — Loss & Learning Rate", color="#e6edf3", fontsize=14, fontweight="bold")
fig.legend(loc="upper right", bbox_to_anchor=(0.92, 0.88), facecolor="#161b22", edgecolor="#30363d",
           labelcolor="#e6edf3", fontsize=10)
plt.tight_layout()
plt.show()

# --- GPU Memory Bar Chart ---
fig2, ax3 = plt.subplots(figsize=(6, 3))
fig2.patch.set_facecolor("#0d1117")
ax3.set_facecolor("#0d1117")

mem_labels = ["Allocated", "Reserved", "Total"]
mem_vals = [m["gpu_mem_allocated_gb"], m["gpu_mem_reserved_gb"], m["gpu_mem_total_gb"]]
colors = ["#3fb950", "#58a6ff", "#30363d"]
bars = ax3.barh(mem_labels, mem_vals, color=colors, height=0.5, edgecolor="#0d1117")
for bar, val in zip(bars, mem_vals):
    ax3.text(val + 0.1, bar.get_y() + bar.get_height()/2, f"{val:.1f} GB",
             va="center", color="#e6edf3", fontsize=11, fontweight="bold")
ax3.set_xlim(0, m["gpu_mem_total_gb"] * 1.3)
ax3.set_title(f"GPU Memory — {m['gpu_name']}", color="#e6edf3", fontsize=13, fontweight="bold")
ax3.tick_params(colors="#8b949e")
ax3.spines["top"].set_visible(False)
ax3.spines["right"].set_visible(False)
for spine in ax3.spines.values():
    spine.set_color("#30363d")
plt.tight_layout()
plt.show()

# --- HTML Dashboard ---
loss_drop = ""
if len(losses) >= 2:
    pct = (losses[0] - losses[-1]) / losses[0] * 100
    loss_drop = f"{pct:.0f}% drop"

mem_util = m["gpu_mem_allocated_gb"] / m["gpu_mem_total_gb"] * 100

html = f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
            max-width: 780px; margin: 20px 0;">

  <div style="display: grid; grid-template-columns: repeat(4, 1fr); gap: 12px; margin-bottom: 16px;">
    <div style="background: linear-gradient(135deg, #1a2332, #161b22); border: 1px solid #30363d;
                border-radius: 12px; padding: 16px; text-align: center;">
      <div style="color: #8b949e; font-size: 11px; text-transform: uppercase; letter-spacing: 1px;">Throughput</div>
      <div style="color: #58a6ff; font-size: 26px; font-weight: 700; margin: 6px 0;">{m['samples_per_sec']:.1f}</div>
      <div style="color: #8b949e; font-size: 12px;">samples/sec</div>
    </div>
    <div style="background: linear-gradient(135deg, #1a2332, #161b22); border: 1px solid #30363d;
                border-radius: 12px; padding: 16px; text-align: center;">
      <div style="color: #8b949e; font-size: 11px; text-transform: uppercase; letter-spacing: 1px;">Audio Processed</div>
      <div style="color: #3fb950; font-size: 26px; font-weight: 700; margin: 6px 0;">{m['audio_hours_processed']:.1f}h</div>
      <div style="color: #8b949e; font-size: 12px;">{m['total_samples']} clips</div>
    </div>
    <div style="background: linear-gradient(135deg, #1a2332, #161b22); border: 1px solid #30363d;
                border-radius: 12px; padding: 16px; text-align: center;">
      <div style="color: #8b949e; font-size: 11px; text-transform: uppercase; letter-spacing: 1px;">Final Loss</div>
      <div style="color: #f0883e; font-size: 26px; font-weight: 700; margin: 6px 0;">{m['final_loss']:.3f}</div>
      <div style="color: #8b949e; font-size: 12px;">{loss_drop}</div>
    </div>
    <div style="background: linear-gradient(135deg, #1a2332, #161b22); border: 1px solid #30363d;
                border-radius: 12px; padding: 16px; text-align: center;">
      <div style="color: #8b949e; font-size: 11px; text-transform: uppercase; letter-spacing: 1px;">Train Time</div>
      <div style="color: #d2a8ff; font-size: 26px; font-weight: 700; margin: 6px 0;">{m['train_time_sec']:.0f}s</div>
      <div style="color: #8b949e; font-size: 12px;">{m['total_steps']} steps</div>
    </div>
  </div>

  <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 12px;">
    <div style="background: #161b22; border: 1px solid #30363d; border-radius: 12px; padding: 18px;">
      <div style="color: #e6edf3; font-size: 13px; font-weight: 600; margin-bottom: 12px;
                  border-bottom: 1px solid #21262d; padding-bottom: 8px;">Model & Training</div>
      <table style="width:100%; color: #c9d1d9; font-size: 12px; border-spacing: 0 6px;">
        <tr><td style="color:#8b949e;">Model</td><td style="text-align:right; font-weight:600;">{m['model']}</td></tr>
        <tr><td style="color:#8b949e;">Task</td><td style="text-align:right;">Speech-to-Text</td></tr>
        <tr><td style="color:#8b949e;">Total params</td><td style="text-align:right;">{m['total_params']/1e6:.0f}M</td></tr>
        <tr><td style="color:#8b949e;">Trainable (LoRA)</td>
            <td style="text-align:right; color:#3fb950;">{m['trainable_params']/1e3:.0f}K ({m['trainable_pct']:.2f}%)</td></tr>
        <tr><td style="color:#8b949e;">Batch size (effective)</td>
            <td style="text-align:right;">{m['batch_size_per_gpu']} x {m['grad_accum_steps']} x {m['num_gpus']}GPU = {m['effective_batch_size']}</td></tr>
        <tr><td style="color:#8b949e;">Learning rate</td><td style="text-align:right;">{m['learning_rate']}</td></tr>
      </table>
    </div>
    <div style="background: #161b22; border: 1px solid #30363d; border-radius: 12px; padding: 18px;">
      <div style="color: #e6edf3; font-size: 13px; font-weight: 600; margin-bottom: 12px;
                  border-bottom: 1px solid #21262d; padding-bottom: 8px;">GPU & Memory</div>
      <table style="width:100%; color: #c9d1d9; font-size: 12px; border-spacing: 0 6px;">
        <tr><td style="color:#8b949e;">GPUs</td><td style="text-align:right; font-weight:600;">{m['num_gpus']}x {m['gpu_name']}</td></tr>
        <tr><td style="color:#8b949e;">VRAM total</td><td style="text-align:right;">{m['gpu_mem_total_gb']} GB per GPU</td></tr>
        <tr><td style="color:#8b949e;">Peak allocated</td>
            <td style="text-align:right; color:#3fb950;">{m['gpu_mem_allocated_gb']} GB ({mem_util:.0f}%)</td></tr>
        <tr><td style="color:#8b949e;">Peak reserved</td><td style="text-align:right;">{m['gpu_mem_reserved_gb']} GB</td></tr>
        <tr><td style="color:#8b949e;">Optimizer</td><td style="text-align:right;">CPU offload (ZeRO-2)</td></tr>
        <tr><td style="color:#8b949e;">Avg step time</td><td style="text-align:right;">{m['avg_step_time_sec']*1000:.0f} ms</td></tr>
      </table>
    </div>
  </div>

  <div style="background: #161b22; border: 1px solid #30363d; border-radius: 12px; padding: 14px 18px;
              margin-top: 12px; display: flex; justify-content: space-between; align-items: center;">
    <span style="color: #8b949e; font-size: 12px;">GPU Memory Utilization</span>
    <div style="flex: 1; margin: 0 16px; background: #21262d; border-radius: 6px; height: 18px; overflow: hidden;">
      <div style="width: {mem_util:.0f}%; height: 100%; border-radius: 6px;
                  background: linear-gradient(90deg, #238636, #3fb950);"></div>
    </div>
    <span style="color: #3fb950; font-size: 13px; font-weight: 700;">{mem_util:.0f}%</span>
  </div>

</div>
"""
display(HTML(html))

## Test: Transcribe Audio

In [ ]:
import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from peft import PeftModel
from datasets import load_dataset, Audio
from IPython.display import Audio as IPAudio, display, HTML

# Load fine-tuned model
base_model = WhisperForConditionalGeneration.from_pretrained(
    "openai/whisper-small", dtype=torch.bfloat16
).to("cuda")
model = PeftModel.from_pretrained(base_model, "./whisper_output/final")
model.eval()
processor = WhisperProcessor.from_pretrained("./whisper_output/final")

# Grab test audio from FLEURS
test_ds = load_dataset("google/fleurs", "en_us", split="test")
test_ds = test_ds.select(range(3))
test_ds = test_ds.cast_column("audio", Audio(sampling_rate=16000))

print("Transcription Results")
print("=" * 60)
for i, ex in enumerate(test_ds):
    audio = ex["audio"]
    inputs = processor(audio["array"], sampling_rate=16000, return_tensors="pt").to("cuda")

    with torch.no_grad():
        predicted_ids = model.generate(**inputs, max_new_tokens=128)

    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    reference = ex["transcription"]

    print(f"\nSample {i+1}:")
    display(IPAudio(audio["array"], rate=16000))
    print(f"  Reference:    {reference}")
    print(f"  Transcription: {transcription}")

# WER score
try:
    from jiwer import wer
    refs = [ex["transcription"] for ex in test_ds]
    preds = []
    for ex in test_ds:
        inputs = processor(ex["audio"]["array"], sampling_rate=16000, return_tensors="pt").to("cuda")
        with torch.no_grad():
            ids = model.generate(**inputs, max_new_tokens=128)
        preds.append(processor.batch_decode(ids, skip_special_tokens=True)[0])
    score = wer(refs, preds)
    print(f"\nWord Error Rate (WER) on {len(refs)} test samples: {score:.1%}")
except ImportError:
    print("\nInstall jiwer for WER: pip install jiwer")

---

## How It Works

```
accelerate launch --num_processes=N train_audio.py
        |
   Worker 0 (GPU 0)       Worker 1 (GPU 1)
        |                       |
   Whisper-small            Whisper-small
   [Mel Encoder]           [Mel Encoder]
   [Text Decoder]          [Text Decoder]
        |                       |
   LoRA on q_proj, v_proj  LoRA on q_proj, v_proj
        |                       |
   DeepSpeed ZeRO-2: optimizer sharded + CPU offload
```

### Key Differences from Text Training

| Aspect | Text (GPT-2) | Audio (Whisper) |
|--------|-------------|----------------|
| **Input** | Token IDs | Mel spectrogram (80x3000) |
| **Output** | Next token | Transcribed text tokens |
| **Architecture** | Decoder-only | Encoder-decoder |
| **Processor** | Tokenizer | WhisperProcessor (audio + text) |
| **Data collator** | Language modeling | Custom padding for audio + labels |
| **Dataset** | Text corpus | Google FLEURS (free) |
| **Metric** | Loss / perplexity | WER (Word Error Rate) |

Everything else is identical: LoRA, DeepSpeed ZeRO-2, Accelerate launcher.

| Platform | GPUs | Cost |
|----------|------|------|
| **Kaggle** | 2x T4 | Free (30h/week) |
| **Colab** | 1x T4 | Free |